# From Predicting to Deciding: Your First Reinforcement Learning Agent

Every notebook so far has been the same kind of problem: given an image, predict the correct label. That's **supervised learning** - there's always a correct answer to learn from.

**Reinforcement learning (RL) is a different problem entirely.** There's no "correct label." Instead, an **agent** takes **actions** in an **environment**, and gets a **reward** - and has to figure out, purely through trial and error, which actions lead to more reward over time.

We'll start with **CartPole**: balance a pole upright on a moving cart by pushing it left or right. Deliberately *no images here* - just 4 numbers describing the situation. That lets us focus entirely on the reward/decision-making idea, without also needing a CNN in the mix yet.

**Why this matters for your actual assignment:** Pong - which you'll build a real agent for - combines this exact reward-based decision loop with the CNN pixel-processing you've been training this whole leerlijn. Pong's "state" is the raw pixels of the game screen, exactly the kind of input your CNNs already know how to handle. This notebook is deliberately the simpler half of that puzzle - once this idea clicks, adding a CNN on top for pixel input is a smaller step than it looks.


In [ ]:
!pip install -q gymnasium
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
print("Observation space:", env.observation_space)
print("  (4 numbers: cart position, cart velocity, pole angle, pole angular velocity)")
print("Action space:", env.action_space)
print("  (2 actions: 0 = push left, 1 = push right)")


## The agent-environment loop

Every RL problem follows the same shape: **observe the state -> choose an action -> receive a reward -> move to a new state -> repeat**, until the episode ends (here: the pole falls over, or the cart drives off the edge).

Each step the pole stays upright is worth **+1 reward** - so the total reward for an episode is just "how many steps did it survive."

Let's wrap that loop in a function, and start with the simplest possible agent: one that ignores the state completely and just picks a random action every time. This becomes our baseline to beat.


In [ ]:
def run_episode(policy_fn):
    state, info = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = policy_fn(state)
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    return total_reward

def random_policy(state):
    return env.action_space.sample()

random_rewards = [run_episode(random_policy) for _ in range(20)]
print("Random policy - average reward over 20 episodes:", np.mean(random_rewards))
print("(CartPole-v1 caps an episode at 500 steps - a trained agent should get close to that)")


## A tiny hand-crafted policy

Random guessing gets nowhere - but we don't need a trained network to do better. Even one line of common sense helps: **if the pole is leaning right, push right to catch it; if it's leaning left, push left.**


In [ ]:
def heuristic_policy(state):
    cart_position, cart_velocity, pole_angle, pole_angular_velocity = state
    return 1 if pole_angle > 0 else 0

heuristic_rewards = [run_episode(heuristic_policy) for _ in range(20)]
print("Heuristic policy - average reward over 20 episodes:", np.mean(heuristic_rewards))
print("Random policy    - average reward over 20 episodes:", np.mean(random_rewards))


### From hand-crafted rules to a learned policy

That one-line rule almost certainly beat random by a wide margin - which is exactly the point. A real reinforcement learning algorithm doesn't need us to hand-write that rule at all. Instead, it **learns its own policy from experience**: try actions, observe the rewards, and gradually adjust which actions to prefer in which states - using ideas like the **Bellman equation** and **Q-learning** from your course book, or policy-gradient methods.

For your actual assignment - **Pong** - the same loop applies, but:
- the **state** is the raw pixel image of the game screen, not 4 clean numbers, so the agent needs a **CNN** (exactly what you've been training throughout this leerlijn) to make sense of it,
- and instead of a hand-written heuristic, the agent needs to **learn** its policy through trial and error, the way Q-learning or a policy-gradient method does.

Put together, that's the whole idea: **CNN (to understand the pixels) + reinforcement learning (to decide what to do about them) = an agent that learns to play Pong from scratch.**
